# **Naive RAG**
The Naive RAG is the simplest technique in the RAG ecosystem, providing a straightforward approach to combining retrieved data with LLM models for efficient user responses.

Research Paper: [RAG](https://arxiv.org/pdf/2005.11401)

## **Initial Setup**

In [ ]:
# Install dependencies.
%pip install -qU faiss-cpu sentence-transformers torch ragas pypdf langchain langchain-community langchain-huggingface langchain-groq langchain-core

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\WORKSPACE\rag-cookbooks\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


## **Indexing**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="AITeamVN/Vietnamese_Embedding_v2",
    model_kwargs={"device": device}
)

embeddings._client.max_seq_length = 2048

d:\WORKSPACE\rag-cookbooks\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\WORKSPACE\rag-cookbooks\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--AITeamVN--Vietnamese_Embedding_v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate

KeyboardInterrupt: 

In [ ]:
# load data
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/bao_cao.pdf")
documents = loader.load()

In [ ]:
# split documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
documents = text_splitter.split_documents(documents)

## **FAISS Vector Database**

In [ ]:
# create FAISS vectorstore
from langchain_community.vectorstores import FAISS
import faiss
import torch

vectorstore = FAISS.from_documents(documents, embeddings)

In [ ]:
# optional: save FAISS index to local disk
import faiss

vectorstore.save_local("faiss_index")

In [ ]:
# optional: load FAISS index from local disk
from langchain_community.vectorstores import FAISS
import faiss
import torch

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

## **FAISS (Optional Tips)**

In [ ]:
# optional: quick FAISS retrieval check
docs = vectorstore.similarity_search("Tên đề tài là gì?", k=2)
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc.page_content[:160]}")

## **Retriever**

In [ ]:
# create retriever
retriever = vectorstore.as_retriever()

## **RAG Chain**

In [ ]:
import os
from langchain_groq import ChatGroq

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("Please set GROQ_API_KEY in your environment before running this cell.")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
# create document chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """
You are a helpful assistant that answers questions based on the provided context.
Use the provided context to answer the question.
Question: {input}
Context: {context}
Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever,  "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# response
response = rag_chain.invoke("Tên đề tài này là gì?")
response

'World War I ended on November 11, 1918.'

In [ ]:
from langchain_core.runnables import RunnableParallel

# 1. Tạo một bước đóng gói input và context
retrieval_setup = RunnableParallel(
    {"context": retriever, "input": RunnablePassthrough()}
)

# 2. Xây dựng chain mới: giữ nguyên context & input, gán thêm kết quả llm vào "answer"
rag_chain_with_sources = retrieval_setup.assign(
    answer=(
        prompt
        | llm
        | StrOutputParser()
    )
)

# 3. Chạy chain
query = "Tên đề tài này là gì?"
response = rag_chain_with_sources.invoke(query)

print("💡 CÂU TRẢ LỜI CỦA LLM:")
print(response["answer"])

print("\n" + "="*50 + "\n")

print("📚 CÁC CHUNK TÀI LIỆU (NGUỒN) ĐÃ SỬ DỤNG:")
for i, doc in enumerate(response["context"], 1):
    print(f"\n[Chunk {i}]")
    print(doc.page_content)

## **Preparing Data for Evaluation**

In [ ]:
question = ["Tên đề tài này là gì?"]
response = []
contexts = []

for query in question:
    result = rag_chain_with_sources.invoke(query)

    response.append(result["answer"])
    contexts.append([doc.page_content for doc in result["context"]])

# To dict
data = {
    "query": question,
    "response": response,
    "context": contexts,
}

print(data)

In [ ]:
# create dataset
from datasets import Dataset
dataset = Dataset.from_dict(data)

In [ ]:
# create dataframe
import pandas as pd
df = pd.DataFrame(dataset)

In [ ]:
df

,query,response,context
0,when did ww1 end?,"World War I ended on November 11, 1918.",[context: ['World War I or the First World War...


In [ ]:
# Convert to dictionary
df_dict = df.to_dict(orient='records')

# Convert context to list
for record in df_dict:
    if not isinstance(record.get('context'), list):
        if record.get('context') is None:
            record['context'] = []
        else:
            record['context'] = [record['context']]

## **Evaluation with RAGAS**

Su dung RAGAS de cham diem chat luong RAG theo cac metric pho bien nhu faithfulness, answer relevancy, context precision va context recall.

In [ ]:
# prepare ragas dataset
from datasets import Dataset

ground_truth = ["Ứng dụng kiến trúc RAG kết hợp LLMs trong xây dựng Chatbot hỏi đáp lịch sử phục vụ giáo dục phổ thông."] * len(question)

ragas_data = {
    "question": question,
    "answer": response,
    "contexts": contexts,
    "ground_truth": ground_truth,
}

ragas_dataset = Dataset.from_dict(ragas_data)
ragas_dataset

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import pandas as pd

try:
    from ragas.metrics import answer_relevancy
except ImportError:
    from ragas.metrics import answer_relevance as answer_relevancy

ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

print("Đang chạy Ragas Evaluation...")
ragas_result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_emb,
)

In [ ]:
# show ragas scores
df_result = ragas_result.to_pandas()

You can view your dataset at: https://app.athina.ai/develop/80872384-24ac-4ad9-824d-74dc02cb7cca


,query,context,response,expected_response,display_name,failed,grade_reason,runtime,model,passed
0,when did ww1 end?,"[context: ['World War I or the First World War (28 July 1914 – 11 November 1918), often abbreviated as WWI, was one of the deadliest global conflicts in history. It was fought between two coalitions, the Allies and the Central Powers. Fighting occurred throughout Europe, the Middle East, Africa, the Pacific, and parts of Asia. An estimated 9 million soldiers were killed in combat, plus another 23 million wounded, while 5 million civilians died as a result of military action, hunger, and dise...","World War I ended on November 11, 1918.",None,Does Response Answer Query,False,"The response directly answers the user's query by providing the specific date on which World War I ended, which is November 11, 1918. This sufficiently covers all aspects of the user's query.",787,gpt-4o,1.0
